In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../../')))

import numpy as np
import pandas as pd
import networkx as nx
import time
import warnings
warnings.filterwarnings('ignore')

from src.graph_utils import cut_value
from src.classical_solvers import exact_brute_force, greedy_one_exchange, simulated_annealing
from src.qubo_builder import build_ising, QUBOParams
from src.qaoa_model import run_qaoa

print("Imports complete.")

Imports complete.


In [2]:
# ─────────────────────────────────────────────
# Load Problem B Graph (Required for Layer 2)
# ─────────────────────────────────────────────

import pandas as pd
import networkx as nx

dfB = pd.read_parquet('../../data/problemB.parquet')

def df_to_graph(df):
    G = nx.Graph()
    for _, r in df.iterrows():
        G.add_edge(int(r.node_1), int(r.node_2), weight=float(r.weight))
    return G

GB = df_to_graph(dfB)

print("Problem B loaded.")
print("Nodes:", GB.number_of_nodes())
print("Edges:", GB.number_of_edges())

Problem B loaded.
Nodes: 180
Edges: 226


In [3]:
import json
from pathlib import Path

LOAD_PATH = Path("../../results/problemB_partition/partition_data.json")

with open(LOAD_PATH, "r") as f:
    partition_data = json.load(f)

clusters = [set(c) for c in partition_data["clusters"]]
edges_per_cluster = partition_data["edges_per_cluster"]
selected_indices = partition_data["selected_indices"]

print("Loaded frozen partition.")
print("Total clusters:", len(clusters))

Loaded frozen partition.
Total clusters: 16


In [4]:
print(selected_indices)
print([edges_per_cluster[i] for i in selected_indices])

[11, 6, 5]
[10, 9, 9]


In [5]:
cluster_results = []
global_partition = {}

for idx, cluster_nodes in enumerate(clusters):

    print(f"\nCluster {idx} — size {len(cluster_nodes)}")

    subG = GB.subgraph(cluster_nodes).copy()

    # Exact optimum (≤12 nodes → safe)
    exact_val = exact_brute_force(subG).cut_value

    model = build_ising(subG, QUBOParams.pure_maxcut())

    result = run_qaoa(
        subG,
        model,
        p=1,
        exact_optimum=exact_val,
        shots=1024,
        n_restarts=3,
        save_path=None
    )

    cluster_results.append({
        "cluster_id": idx,
        "nodes": list(cluster_nodes),
        "cut": result.best_cut_value,
        "ratio": result.approx_ratio,
        "gamma": result.optimal_gamma,
        "beta": result.optimal_beta
    })

    # Extract best bitstring
    from src.qaoa_model import _best_cut_from_counts
    best_cut, best_part = result.best_cut_value, None

    # Reconstruct best partition from saved counts
    # (Use the most probable bitstring)
    best_bitstring = max(result.bitstring_counts, key=result.bitstring_counts.get)
    nodes_list = model.nodes

    spins = {nodes_list[i]: (1 if best_bitstring[::-1][i]=='0' else -1)
             for i in range(len(nodes_list))}

    global_partition.update(spins)


Cluster 0 — size 11
  QAOA p=1  |  11 qubits  |  55 ZZ terms
  grid search (p=1)... best_E = 183.1541
  COBYLA (3 restarts)... best_E = 174.4726
  sampling (1024 shots)... best cut = 395.2485

Cluster 1 — size 11
  QAOA p=1  |  11 qubits  |  55 ZZ terms
  grid search (p=1)... best_E = 152.8282
  COBYLA (3 restarts)... best_E = 188.8312
  sampling (1024 shots)... best cut = 431.9207

Cluster 2 — size 11
  QAOA p=1  |  11 qubits  |  55 ZZ terms
  grid search (p=1)... best_E = 192.8764
  COBYLA (3 restarts)... best_E = 190.4817
  sampling (1024 shots)... best cut = 502.1472

Cluster 3 — size 12
  QAOA p=1  |  12 qubits  |  66 ZZ terms
  grid search (p=1)... best_E = 176.2799
  COBYLA (3 restarts)... best_E = 178.0756
  sampling (1024 shots)... best cut = 436.0249

Cluster 4 — size 11
  QAOA p=1  |  11 qubits  |  55 ZZ terms
  grid search (p=1)... best_E = 197.6283
  COBYLA (3 restarts)... best_E = 200.3524
  sampling (1024 shots)... best cut = 496.2168

Cluster 5 — size 11
  QAOA p=1  | 

In [6]:
clustered_cut = cut_value(GB, global_partition)
print("Clustered Global Cut:", clustered_cut)

Clustered Global Cut: 5346.621257536227


In [8]:
print("Total cluster results:", len(cluster_results))
print("Total spins assigned:", len(global_partition))

Total cluster results: 16
Total spins assigned: 180


In [7]:
print("\nRunning classical baselines...")

# Greedy
greedy_res = greedy_one_exchange(GB)
print("Greedy cut:", greedy_res.cut_value)

# Simulated Annealing
anneal_res = simulated_annealing(GB, max_iter=20000)
print("Simulated Annealing cut:", anneal_res.cut_value)

# Random baseline
def random_cut(G, trials=20):
    best = 0
    for _ in range(trials):
        spins = {n: np.random.choice([-1,1]) for n in G.nodes()}
        best = max(best, cut_value(G, spins))
    return best

random_best = random_cut(GB)
print("Random baseline (best of 20):", random_best)


Running classical baselines...
Greedy cut: 6778.186721732447
Simulated Annealing cut: 6591.012769702316
Random baseline (best of 20): 4405.407625364018


In [9]:
def one_exchange_from_partition(G, initial_spins, max_iter=5):
    """
    Perform local 1-flip improvement starting from given partition.
    """
    spins = initial_spins.copy()
    nodes = list(G.nodes())
    
    def delta_flip(node):
        delta = 0
        for nb in G.neighbors(node):
            w = G[node][nb]['weight']
            if spins[node] == spins[nb]:
                delta += w
            else:
                delta -= w
        return delta
    
    for _ in range(max_iter):
        improved = False
        for node in nodes:
            gain = delta_flip(node)
            if gain > 0:
                spins[node] *= -1
                improved = True
        if not improved:
            break
    
    return spins

In [10]:
refined_partition = one_exchange_from_partition(GB, global_partition)
refined_cut = cut_value(GB, refined_partition)

delta = refined_cut - clustered_cut

print("Clustered Cut:", clustered_cut)
print("Refined Cut:", refined_cut)
print("Improvement Δ:", delta)

Clustered Cut: 5346.621257536227
Refined Cut: 6435.604385869069
Improvement Δ: 1088.9831283328422


In [11]:
best_classical = greedy_res.cut_value

r_clustered = clustered_cut / best_classical
r_refined = refined_cut / best_classical

print("Clustered ratio vs greedy:", r_clustered)
print("Refined ratio vs greedy:", r_refined)

Clustered ratio vs greedy: 0.788798166387732
Refined ratio vs greedy: 0.9494581146953389


In [12]:
# ─────────────────────────────────────────────
# Noise Simulation (Calibrated to Ankaa-3)
# ─────────────────────────────────────────────

import numpy as np
import pandas as pd

# --- Frozen Hardware Parameters ---
F_median = 0.9852366338446036
T2_median = 22.07562115774664   # microseconds
iswap_duration_us = 0.2        # 200 ns per iSWAP


def predict_noisy_cluster_cut(ideal_cut, num_edges):
    """
    Predict degraded cluster cut using hardware survival model.

    Survival model:
        S = F^N2Q * exp(-T_circuit / T2)

    where:
        N2Q = 2 * edges  (approximate iSWAP usage per ZZ)
        T_circuit ≈ N2Q * iswap_duration
    """

    N2Q = num_edges * 2
    coherence_factor = np.exp(-(N2Q * iswap_duration_us) / T2_median)
    gate_survival = F_median ** N2Q
    survival = gate_survival * coherence_factor

    noisy_cut = ideal_cut * survival

    return noisy_cut, survival, N2Q


# --- Compute predictions for all clusters ---
noisy_results = []

for r in cluster_results:

    ideal_cut = r["cut"]
    subG = GB.subgraph(r["nodes"])
    num_edges = subG.number_of_edges()

    noisy_cut, survival, N2Q = predict_noisy_cluster_cut(
        ideal_cut,
        num_edges
    )

    noisy_results.append({
        "cluster_id": r["cluster_id"],
        "edges": num_edges,
        "native_iSWAP": N2Q,
        "ideal_cluster_cut": ideal_cut,
        "predicted_survival": survival,
        "predicted_noisy_cluster_cut": noisy_cut
    })


# --- Convert to DataFrame for clean display ---
noisy_df = pd.DataFrame(noisy_results)

print("\nPredicted Hardware-Degraded Cluster Performance:")
display(noisy_df.round(4))


Predicted Hardware-Degraded Cluster Performance:


,cluster_id,edges,native_iSWAP,ideal_cluster_cut,predicted_survival,predicted_noisy_cluster_cut
0,0,11,22,395.2485,0.5907,233.4538
1,1,11,22,431.9207,0.5907,255.1143
2,2,11,22,502.1472,0.5907,296.5936
3,3,14,28,436.0249,0.5116,223.0888
4,4,12,24,496.2168,0.5630,279.3921
5,5,9,18,295.4591,0.6500,192.0456
6,6,9,18,375.8393,0.6500,244.2920
7,7,10,20,340.2948,0.6196,210.8503
8,8,11,22,343.3757,0.5907,202.8151
9,9,9,18,283.4958,0.6500,184.2696


In [13]:
predicted_global_cut = sum(noisy_df["predicted_noisy_cluster_cut"])
print("Predicted degraded global cut:", predicted_global_cut)

Predicted degraded global cut: 3602.9212078988885


## Interpretation: Predicted Hardware-Degraded Cluster Performance

Using the calibrated Ankaa-3 noise model:

$
S = F^{N_{2Q}} \cdot \exp\left(-\frac{T_{\text{circuit}}}{T_2}\right),
\quad N_{2Q} = 2 \times \text{edges}
$

we estimated hardware survival and degraded cluster cut values.

### Key Observations

1. **Survival decreases monotonically with native iSWAP count**, as expected from the feasibility envelope:
   - Clusters with 18 iSWAP → survival ≈ 0.65  
   - Clusters with 22 iSWAP → survival ≈ 0.59  
   - Clusters with 28 iSWAP → survival ≈ 0.51  

   This is consistent with the exponential decay predicted by the feasibility envelope derived earlier

2. **Under this first-order survival model, raw stitched clustered performance is expected to degrade substantially.**  
   Summing predicted degraded cluster cuts yields a global value ≈ 3600–3700,  
   compared to:
   - Ideal clustered: 5346  
   - Greedy baseline: 6778  

   This indicates that *pure stitched clustered QAOA* is not expected to outperform classical baselines under realistic hardware noise.

3. **Important structural observation:**  
   Higher survival does not necessarily imply higher noisy cut.  
   For example:
   - A 28-iSWAP cluster (survival ≈ 0.51) may still produce higher noisy cut than an 18-iSWAP cluster (survival ≈ 0.65), because ideal cluster cut magnitudes differ.

   Therefore, hardware validation must examine **survival trends independently from objective magnitude.**

---

### Architectural Implication

The results reinforce the intended design philosophy:

- Hardware constraints determine feasible cluster size.
- Clustered QAOA produces structured but imperfect global solutions.
- Lightweight classical refinement (one_exchange) recovers much of the lost performance.
- This naturally yields a **hardware-constrained quantum-classical hybrid optimization architecture**, rather than a pure quantum advantage claim.

This interpretation will be experimentally validated in Layer 3 using percentile-selected clusters on Ankaa-3.

In [14]:
import json
from pathlib import Path

SAVE_PATH = Path("../../results/problemB_qaoa_layer2")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

layer2_data = {
    "clusters": [list(c) for c in clusters],
    "selected_indices": selected_indices,
    "cluster_results": cluster_results,
    "clustered_cut": clustered_cut,
    "refined_cut": refined_cut,
    "greedy_cut": greedy_res.cut_value,
    "annealing_cut": anneal_res.cut_value
}

with open(SAVE_PATH / "layer2_results.json", "w") as f:
    json.dump(layer2_data, f, indent=4)

print("Layer 2 results saved.")

Layer 2 results saved.


In [15]:
# ─────────────────────────────────────────────
# Layer 2 Saved Data Integrity Check
# ─────────────────────────────────────────────

import json
from pathlib import Path
import numpy as np

LOAD_PATH = Path("../../results/problemB_qaoa_layer2/layer2_results.json")

assert LOAD_PATH.exists(), "Layer 2 results file not found!"

with open(LOAD_PATH, "r") as f:
    loaded_data = json.load(f)

print("Layer 2 file loaded successfully.\n")

# --- Basic Structure Checks ---
required_keys = [
    "clusters",
    "selected_indices",
    "cluster_results",
    "clustered_cut",
    "refined_cut",
    "greedy_cut",
    "annealing_cut"
]

for key in required_keys:
    assert key in loaded_data, f"Missing key: {key}"

print("All required keys present.")

# --- Cluster Count Check ---
clusters_loaded = loaded_data["clusters"]
cluster_results_loaded = loaded_data["cluster_results"]

print("Total clusters saved:", len(clusters_loaded))
print("Total cluster results saved:", len(cluster_results_loaded))

assert len(clusters_loaded) == 16, "Expected 16 clusters."
assert len(cluster_results_loaded) == 16, "Expected 16 cluster results."

# --- Selected Indices Check ---
selected_loaded = loaded_data["selected_indices"]
print("Selected hardware clusters:", selected_loaded)

assert len(selected_loaded) == 3, "Expected 3 selected hardware clusters."

# --- Parameter Presence Check ---
for r in cluster_results_loaded:
    assert "gamma" in r and "beta" in r, f"Missing gamma/beta in cluster {r['cluster_id']}"
    assert "cut" in r and "ratio" in r, f"Missing cut/ratio in cluster {r['cluster_id']}"

print("All cluster parameters present.")

# --- Metric Consistency ---
print("\nStored Metrics:")
print("Clustered Cut:", loaded_data["clustered_cut"])
print("Refined Cut:", loaded_data["refined_cut"])
print("Greedy Cut:", loaded_data["greedy_cut"])
print("Annealing Cut:", loaded_data["annealing_cut"])

print("\nIntegrity check PASSED. Layer 2 is hardware-ready.")

Layer 2 file loaded successfully.

All required keys present.
Total clusters saved: 16
Total cluster results saved: 16
Selected hardware clusters: [11, 6, 5]
All cluster parameters present.

Stored Metrics:
Clustered Cut: 5346.621257536227
Refined Cut: 6435.604385869069
Greedy Cut: 6778.186721732447
Annealing Cut: 6591.012769702316

Integrity check PASSED. Layer 2 is hardware-ready.


In [16]:
print("\nFrozen γ, β parameters per cluster:\n")

for r in loaded_data["cluster_results"]:
    
    gamma = r["gamma"]
    beta = r["beta"]
    
    # Handle list vs scalar
    if isinstance(gamma, list):
        gamma_val = gamma[0]
    else:
        gamma_val = gamma
        
    if isinstance(beta, list):
        beta_val = beta[0]
    else:
        beta_val = beta

    print(
        f"Cluster {r['cluster_id']:2d} | "
        f"gamma = {gamma_val:.6f} | "
        f"beta = {beta_val:.6f}"
    )


Frozen γ, β parameters per cluster:

Cluster  0 | gamma = 0.414496 | beta = 1.264744
Cluster  1 | gamma = 2.961461 | beta = 0.423141
Cluster  2 | gamma = 2.793751 | beta = 1.139883
Cluster  3 | gamma = 0.414471 | beta = 1.229363
Cluster  4 | gamma = 2.472860 | beta = 1.168193
Cluster  5 | gamma = 2.694297 | beta = 1.164952
Cluster  6 | gamma = 3.019918 | beta = 0.356015
Cluster  7 | gamma = 2.947035 | beta = 0.404237
Cluster  8 | gamma = 2.026973 | beta = 0.406455
Cluster  9 | gamma = 2.373641 | beta = 0.419983
Cluster 10 | gamma = 3.195311 | beta = 1.096552
Cluster 11 | gamma = 2.996446 | beta = 0.446993
Cluster 12 | gamma = 0.531093 | beta = 1.965650
Cluster 13 | gamma = -0.025046 | beta = 0.407226
Cluster 14 | gamma = 3.049007 | beta = 1.985686
Cluster 15 | gamma = 2.874123 | beta = 2.344805


In [17]:
total_nodes = set()
for c in clusters_loaded:
    total_nodes.update(c)

assert len(total_nodes) == 180, "Node coverage mismatch!"
print("All 180 nodes covered.")

All 180 nodes covered.
